# 📊 10 — Evaluate: Combined JD Model (gen + edit adapters)

Evaluates **`09_combined_fixed`** — gemma-2b-it base with two LoRA adapters:
- **`gen` adapter** → raw text → structured JSON
- **`edit` adapter** → existing JSON + instruction → updated JSON

### Metrics covered (Problem Statement requirements)
| Metric | Adapter |
|---|---|
| JSON Validity Rate | gen + edit |
| Field Coverage Rate | gen + edit |
| Structural Completeness | gen + edit |
| ROUGE-1 / ROUGE-2 / ROUGE-L | gen + edit |
| BLEU | gen + edit |
| Perplexity | gen (representative) |
| Target Field Accuracy | edit only |
| Training/Val loss curves | loaded from log_history |
| Version comparison | across saved metrics files |

### Shared utilities
All metric functions live in **`jd_eval_utils.py`** and are also importable from `09_combined_fixed`.

---
## 0. GPU Check

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ CUDA not available"
torch.cuda.set_device(0)
print("✅ GPU:", torch.cuda.get_device_name(0))
print(f"   Total VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   Free  VRAM : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9:.1f} GB")

---
## 1. Install / Import Dependencies

In [ ]:
# Run once if packages are missing
# !pip install -q evaluate rouge-score bert-score

In [ ]:
import json
import math
import glob
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ── Shared evaluation utilities (jd_eval_utils.py must be in the same folder) ──
from jd_eval_utils import (
    # JSON helpers
    extract_first_json, enforce_input_constraints, jd_to_text,
    # Structural metrics
    json_validity_rate, field_coverage_rate,
    structural_completeness, target_field_accuracy,
    # Text metrics
    compute_bleu, compute_rouge, compute_perplexity,
    # Inference wrapper
    generate_output,
    # Benchmark runner & display
    run_benchmark, print_combined_summary,
    # Version persistence
    save_metrics, load_all_versions,
    # Constants
    SCHEMA_FIELDS,
)

print("✅ All imports OK")

---
## 2. Load Base Model + Both Adapters
Identical to `09_combined_fixed` — one base model, two LoRA adapters, no extra VRAM.

In [ ]:
BASE_MODEL   = "models/gemma-2b-it"
GEN_ADAPTER  = "./models/gemma-2b-it-fine-tuned"
EDIT_ADAPTER = "./models/gemma-2b-it-fine-tuned-edit"
RUN_NAME     = "combined-fixed-eval"   # tag for saved metrics file

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

# Register both adapters exactly as in 09_combined_fixed
model = PeftModel.from_pretrained(base, GEN_ADAPTER, adapter_name="gen")
model.load_adapter(EDIT_ADAPTER, adapter_name="edit")

model.eval()
torch.set_grad_enabled(False)

print("✅ Model loaded. Adapters: 'gen', 'edit'")
print(f"   Device : {next(model.parameters()).device}")
print(f"   Dtype  : {next(model.parameters()).dtype}")

---
## 3. Prompts — Exact Copy from 09_combined_fixed
Keeping prompts in one place here so the eval always matches the training format.

In [ ]:
SYSTEM_GEN = """You are an AI system designed to transform raw, unstructured job descriptions into structured, professional, and ATS-friendly job descriptions.

Your task:
- Extract AND rewrite content into a polished, professional format.
- Expand short or vague statements into clear, detailed, and actionable bullet points.
- Improve grammar, clarity, and tone while preserving original meaning.

Rules:
- Output MUST be valid JSON only.
- Follow the exact schema provided.
- Do NOT include explanations or extra text.
- Do NOT hallucinate unrealistic details.
- Do NOT copy sentences directly from input — always rewrite them professionally.

Enhancement Rules:
- Convert short phrases into complete, professional sentences.
- Add clarity by specifying intent.
- Use strong action verbs (manage, ensure, deliver, coordinate, analyze).
- Maintain ATS-friendly language with relevant keywords.
- Avoid vague wording like 'do', 'work on', 'handle'.
- Only include information explicitly present in the input.
- Do NOT infer industry or qualifications unless clearly mentioned.

Writing Style:
- Use concise but complete sentences.
- Each bullet point should be meaningful and self-contained.
- Maintain consistency across all sections.
- Generate ONLY ONE JSON object.
- Stop immediately after closing }."""

OUTPUT_SCHEMA = """
{
  "job_title": "",
  "location": "",
  "industry": "",
  "responsibilities": [],
  "requirements": [],
  "qualifications": [],
  "experience": [],
  "other_requirements": []
}
"""

def gen_prompt(content):
    return f"""### SYSTEM:
{SYSTEM_GEN}

### USER:
Convert the following raw job description into structured JSON.

### Expected OUTPUT FORMAT:
Return a fully populated JSON following this schema:
{OUTPUT_SCHEMA}

### INPUT:
{content}

### RESPONSE:
"""


def edit_prompt(current_json: str, instruction: str):
    return f"""
### SYSTEM:
You are an AI system designed to MODIFY an existing job description JSON.

Your task:
- Update the given JSON based ONLY on the user instruction.
- Do NOT regenerate the entire job description.
- Make ONLY the necessary changes.

---

RULES:
- Output MUST be valid JSON only.
- Return ONLY ONE JSON object.
- Do NOT include explanations or extra text.
- Do NOT change fields that are not related to the instruction.
- Preserve all existing data unless modification is required.
- Do NOT hallucinate new fields or unnecessary content.

---

EDITING GUIDELINES:

1. ADD: Add new items to the correct field without removing existing ones.
2. REMOVE: Remove only the specified content.
3. UPDATE: Modify only the specified field value.
4. REPLACE: Replace only the mentioned parts.
5. REFINE: Improve wording while preserving meaning.
6. IMPROVE: Make content more professional or detailed without changing intent.
7. REGENERATE: Rewrite the entire JSON ONLY if explicitly requested.

---

### USER:

Existing JSON:
{current_json}

Instruction:
{instruction}

---

### RESPONSE:
"""

print("✅ Prompts defined (gen_prompt, edit_prompt)")

---
## 4. Benchmark Test Sets

### 4a. Generation Test Set
5 diverse raw JD inputs with human-written reference outputs.
Extend with your annotated dataset for production-grade scores.

In [ ]:
GEN_TEST_CASES = [
    {
        "input": "we need python dev.. 2+ yrs exp!!! backend work",
        "reference": {
            "job_title": "Python Developer",
            "location": "",
            "industry": "",
            "responsibilities": [
                "Design and develop backend services and APIs using Python.",
                "Write clean, maintainable, and well-documented code.",
                "Collaborate with team members to deliver features on schedule."
            ],
            "requirements": [
                "Minimum 2 years of experience in Python development.",
                "Proficiency in backend frameworks such as Django or FastAPI."
            ],
            "qualifications": "",
            "experience": ["2+ years of Python backend development"],
            "other_requirements": []
        }
    },
    {
        "input": "hiring data analyst. must know excel and sql. freshers ok. delhi office.",
        "reference": {
            "job_title": "Data Analyst",
            "location": "Delhi",
            "industry": "",
            "responsibilities": [
                "Analyze data using Excel and SQL to generate actionable business insights.",
                "Create and maintain dashboards and reports for stakeholders.",
                "Identify trends, patterns, and anomalies in datasets."
            ],
            "requirements": [
                "Proficiency in Microsoft Excel including pivot tables and advanced formulas.",
                "Strong knowledge of SQL for querying relational databases."
            ],
            "qualifications": "",
            "experience": ["Freshers welcome; internship experience preferred"],
            "other_requirements": ["Must be available to work from the Delhi office"]
        }
    },
    {
        "input": "need ML engineer with pytorch, tensorflow. remote. 3-5 yrs. deep learning must.",
        "reference": {
            "job_title": "Machine Learning Engineer",
            "location": "Remote",
            "industry": "",
            "responsibilities": [
                "Design, train, and evaluate deep learning models using PyTorch and TensorFlow.",
                "Optimize and deploy trained models to production environments.",
                "Collaborate with data scientists and engineers to build end-to-end ML pipelines."
            ],
            "requirements": [
                "Expert-level proficiency in PyTorch and TensorFlow.",
                "Deep understanding of neural network architectures and training techniques.",
                "3–5 years of hands-on machine learning or deep learning experience."
            ],
            "qualifications": "",
            "experience": ["3–5 years in machine learning engineering"],
            "other_requirements": []
        }
    },
    {
        "input": "UI/UX Designer needed. Figma + Adobe XD. portfolio required. bangalore. 2 yrs exp.",
        "reference": {
            "job_title": "UI/UX Designer",
            "location": "Bangalore",
            "industry": "",
            "responsibilities": [
                "Design intuitive user interfaces and engaging user experiences for web and mobile products.",
                "Produce high-fidelity wireframes and interactive prototypes using Figma and Adobe XD.",
                "Work closely with product and engineering teams to translate requirements into pixel-perfect designs."
            ],
            "requirements": [
                "Expert proficiency in Figma and Adobe XD.",
                "Strong design portfolio showcasing UI/UX projects.",
                "Minimum 2 years of professional UI/UX design experience."
            ],
            "qualifications": "",
            "experience": ["2 years of UI/UX design experience"],
            "other_requirements": ["Portfolio submission mandatory with application"]
        }
    },
    {
        "input": "DevOps engineer wanted. AWS, docker, kubernetes, CI/CD pipeline. 4+ yrs. Mumbai.",
        "reference": {
            "job_title": "DevOps Engineer",
            "location": "Mumbai",
            "industry": "",
            "responsibilities": [
                "Design, provision, and manage scalable infrastructure on AWS.",
                "Build and maintain CI/CD pipelines to streamline software delivery.",
                "Orchestrate containerised workloads using Docker and Kubernetes.",
                "Monitor system health and respond to infrastructure incidents promptly."
            ],
            "requirements": [
                "Deep expertise in AWS cloud services and infrastructure-as-code.",
                "Hands-on experience with Docker and Kubernetes.",
                "Proven track record of building and maintaining CI/CD pipelines.",
                "Minimum 4 years of DevOps or platform engineering experience."
            ],
            "qualifications": "",
            "experience": ["4+ years of DevOps engineering"],
            "other_requirements": []
        }
    },
]

print(f"✅ Generation test set: {len(GEN_TEST_CASES)} samples")

### 4b. Edit Test Set
Each case starts from a shared base state, applies one instruction, and checks the expected field update.

In [ ]:
# Shared base state — mirrors what the gen adapter actually produced in 09_combined_fixed
BASE_JD = {
    "job_title": "Python Developer",
    "location": "N/A",
    "industry": "",
    "responsibilities": [
        "Develop, test, and maintain high-quality Python applications.",
        "Collaborate with cross-functional teams to define, design, and ship new features.",
        "Write clean, efficient, and well-documented code.",
        "Troubleshoot and debug issues in existing systems.",
        "Participate in code reviews and provide constructive feedback.",
        "Stay updated with emerging technologies and trends in the field."
    ],
    "requirements": [
        "Minimum of 2 years of experience in Python development.",
        "Proficiency in data structures, algorithms, and software engineering principles.",
        "Strong understanding of Object-Oriented Programming concepts.",
        "Experience working with relational databases (e.g., MySQL, PostgreSQL) and NoSQL databases.",
        "Familiarity with cloud platforms such as AWS or Azure is a plus.",
        "Excellent problem-solving and analytical skills."
    ],
    "qualifications": "",
    "experience": ["2+ years of experience in Python development"],
    "other_requirements": ["Willingness to learn and adapt to new technologies quickly"]
}

EDIT_TEST_CASES = [
    {
        "state"          : BASE_JD,
        "instruction"    : "change location to Mandi",
        "expected_field" : "location",
        "expected_value" : "Mandi",
    },
    {
        "state"          : BASE_JD,
        "instruction"    : "change industry to Information Technology",
        "expected_field" : "industry",
        "expected_value" : "Information Technology",
    },
    {
        "state"          : BASE_JD,
        "instruction"    : "add Docker and Kubernetes to requirements",
        "expected_field" : "requirements",
        "expected_value" : "docker",   # substring match, case-insensitive
    },
    {
        "state"          : BASE_JD,
        "instruction"    : "change job title to Senior Python Developer",
        "expected_field" : "job_title",
        "expected_value" : "Senior Python Developer",
    },
    {
        "state"          : BASE_JD,
        "instruction"    : "change location to Hyderabad and add ability to work under pressure to other_requirements",
        "expected_field" : "location",
        "expected_value" : "Hyderabad",
    },
    {
        "state"          : BASE_JD,
        "instruction"    : "add fluency in English and Hindi to other_requirements",
        "expected_field" : "other_requirements",
        "expected_value" : "english",
    },
    {
        "state"          : BASE_JD,
        "instruction"    : "increase minimum experience requirement to 4 years",
        "expected_field" : "experience",
        "expected_value" : "4",
    },
]

print(f"✅ Edit test set: {len(EDIT_TEST_CASES)} samples")

---
## 5. Run Generation Benchmark
Calls `run_benchmark()` from `jd_eval_utils` — activates the `gen` adapter automatically.

In [ ]:
gen_metrics, gen_outputs = run_benchmark(
    model=model,
    tokenizer=tokenizer,
    test_cases=GEN_TEST_CASES,
    adapter_name="gen",
    prompt_fn=gen_prompt,
    max_new_tokens=1536,
    verbose=True,
)

---
## 6. Run Edit Benchmark
Calls `run_benchmark()` with `adapter_name="edit"` — switches adapter automatically.

In [ ]:
edit_metrics, edit_outputs = run_benchmark(
    model=model,
    tokenizer=tokenizer,
    test_cases=EDIT_TEST_CASES,
    adapter_name="edit",
    prompt_fn=edit_prompt,
    max_new_tokens=1024,
    verbose=True,
)

---
## 7. Perplexity (gen adapter — matches SmolLM notebook)
Uses the generation prompts as full texts so we measure how well the model has learned the task.

In [ ]:
model.set_adapter("gen")

# Build full prompt+reference texts to feed into perplexity
perp_texts = [
    gen_prompt(c["input"]) + json.dumps(c["reference"], indent=2)
    for c in GEN_TEST_CASES
]

perplexity = compute_perplexity(
    model=model,
    tokenizer=tokenizer,
    texts=perp_texts,
    max_samples=len(perp_texts),
    max_length=1024,
)

print(f"\n📐 Test Perplexity (gen adapter): {perplexity:.2f}")
gen_metrics["perplexity"] = perplexity

---
## 8. Combined Summary Table

In [ ]:
print_combined_summary(gen_metrics, edit_metrics)

---
## 9. Per-Sample Inspection
Print each generated output alongside its reference for qualitative review.

In [ ]:
print("=" * 70)
print("GEN ADAPTER — Per-Sample Results")
print("=" * 70)
for i, (case, output) in enumerate(zip(GEN_TEST_CASES, gen_outputs)):
    print(f"\n── Sample {i+1} ────────────────────────────")
    print(f"INPUT     : {case['input']}")
    valid = isinstance(output, dict)
    print(f"VALID JSON: {'✅ Yes' if valid else '❌ No'}")
    if valid:
        print(f"job_title : {output.get('job_title', 'N/A')}")
        print(f"location  : {output.get('location',  'N/A')}")
        print(f"industry  : {output.get('industry',  'N/A')}")
        resp = output.get('responsibilities', [])
        print(f"resps     : {len(resp)} items — first: {resp[0][:60] if resp else 'none'}...")

print("\n" + "=" * 70)
print("EDIT ADAPTER — Per-Sample Results")
print("=" * 70)
for i, (case, output) in enumerate(zip(EDIT_TEST_CASES, edit_outputs)):
    field  = case["expected_field"]
    expect = case["expected_value"]
    actual = output.get(field, "") if isinstance(output, dict) else ""
    if isinstance(actual, list):
        hit = any(expect.lower() in str(v).lower() for v in actual)
    else:
        hit = expect.lower() in str(actual).lower()
    status = "✅" if hit else "❌"
    print(f"\n── Sample {i+1} ──  [{status}]")
    print(f"INSTRUCTION : {case['instruction']}")
    print(f"FIELD       : {field}")
    print(f"EXPECTED    : {expect}")
    print(f"GOT         : {actual}")

---
## 10. Training / Validation Loss Curves
Load `log_history` from a saved trainer state or supply manually.
Matches the SmolLM notebook's loss curve cell exactly.

In [ ]:
# ── Option A: load from trainer_state.json saved during training ──────────────
# Adjust path to wherever your training output was saved.
TRAINER_STATE_PATH = "./models/gemma-2b-it-fine-tuned/trainer_state.json"

try:
    with open(TRAINER_STATE_PATH) as f:
        trainer_state = json.load(f)
    log_history = trainer_state["log_history"]
    print(f"✅ Loaded {len(log_history)} log entries from {TRAINER_STATE_PATH}")
except FileNotFoundError:
    print(f"[WARN] {TRAINER_STATE_PATH} not found — using placeholder data.")
    print("       Replace with your actual trainer_state.json to see real curves.")
    # Placeholder so the rest of the cell doesn't crash
    log_history = [
        {"step": s, "loss": 2.5 - s * 0.04} for s in range(1, 11)
    ] + [
        {"step": s, "eval_loss": 2.6 - s * 0.035} for s in [5, 10]
    ]

train_steps  = [x["step"] for x in log_history if "loss"      in x and "eval_loss" not in x]
train_losses = [x["loss"] for x in log_history if "loss"      in x and "eval_loss" not in x]
eval_steps   = [x["step"] for x in log_history if "eval_loss" in x]
eval_losses  = [x["eval_loss"] for x in log_history if "eval_loss" in x]

plt.figure(figsize=(10, 4))
plt.plot(train_steps, train_losses, label="Train Loss", color="steelblue")
if eval_steps:
    plt.plot(eval_steps, eval_losses, label="Val Loss", color="coral", marker="o")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training & Validation Loss — gemma-2b-it (gen adapter)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

if train_losses:
    print(f"Final train loss : {train_losses[-1]:.4f}")
if eval_losses:
    print(f"Final val loss   : {eval_losses[-1]:.4f}")

---
## 11. Save Metrics + Version Comparison
Saves a `metrics_<run_name>.json` and plots improvement across multiple fine-tuning runs,  
matching the SmolLM notebook's version comparison cell exactly.

In [ ]:
# ── Build full metrics dict for this run ─────────────────────────────────────
full_metrics = {
    "model_version"            : RUN_NAME,
    "base_model"               : BASE_MODEL,
    # Gen adapter
    "gen_json_validity"        : gen_metrics["json_validity_rate"],
    "gen_field_coverage"       : gen_metrics["field_coverage"],
    "gen_structural_completeness": gen_metrics["structural_completeness"],
    "gen_valid_json_pct"       : gen_metrics["valid_json_pct"],
    "gen_bleu"                 : gen_metrics["bleu"],
    "gen_rouge1"               : gen_metrics["rouge1"],
    "gen_rouge2"               : gen_metrics["rouge2"],
    "gen_rougeL"               : gen_metrics["rougeL"],
    "gen_perplexity"           : gen_metrics.get("perplexity", None),
    # Edit adapter
    "edit_json_validity"       : edit_metrics["json_validity_rate"],
    "edit_field_coverage"      : edit_metrics["field_coverage"],
    "edit_structural_completeness": edit_metrics["structural_completeness"],
    "edit_target_field_acc"    : edit_metrics.get("target_field_accuracy", None),
    "edit_bleu"                : edit_metrics["bleu"],
    "edit_rouge1"              : edit_metrics["rouge1"],
    "edit_rouge2"              : edit_metrics["rouge2"],
    "edit_rougeL"              : edit_metrics["rougeL"],
    # Loss (if available)
    "train_loss_final"         : train_losses[-1] if train_losses else None,
    "val_loss_final"           : eval_losses[-1]  if eval_losses  else None,
}

save_metrics(full_metrics, RUN_NAME)

print("\n📊 Full Metrics:")
for k, v in full_metrics.items():
    print(f"  {k:<35}: {v}")

In [ ]:
# ── Version comparison plot — matches SmolLM notebook section 6 ───────────────
all_versions = load_all_versions("metrics_*.json")

if len(all_versions) > 1:
    vers         = [m["model_version"]             for m in all_versions]
    perps        = [m.get("gen_perplexity", 0)     for m in all_versions]
    rouge_ls     = [m.get("gen_rougeL", 0)         for m in all_versions]
    completeness = [m.get("gen_structural_completeness", 0) for m in all_versions]
    tfa          = [m.get("edit_target_field_acc", 0) for m in all_versions]

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))

    axes[0].plot(vers, perps, marker="o", color="steelblue")
    axes[0].set_title("Perplexity (lower = better)")
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].plot(vers, rouge_ls, marker="o", color="coral")
    axes[1].set_title("ROUGE-L / Gen (higher = better)")
    axes[1].tick_params(axis="x", rotation=45)

    axes[2].plot(vers, completeness, marker="o", color="green")
    axes[2].set_title("Structural Completeness (higher = better)")
    axes[2].tick_params(axis="x", rotation=45)

    axes[3].plot(vers, tfa, marker="o", color="purple")
    axes[3].set_title("Edit Target Field Accuracy")
    axes[3].tick_params(axis="x", rotation=45)

    plt.suptitle("Model Improvement Across Fine-Tuning Versions", fontsize=13)
    plt.tight_layout()
    plt.savefig("version_comparison.png", dpi=150)
    plt.show()
    print("✅ Saved version_comparison.png")
else:
    print("Only 1 version so far — run more fine-tuning iterations to see comparison.")